# 02. Trie Problems

Typical interview questions built on the basic trie.

**Learning order:** WILDCARD SEARCH → REPLACE WORDS → LONGEST WORD → LONGEST COMMON PREFIX → MAP SUM → INTERVIEW THINKING

## 1. Add and Search Word (with `.` wildcard)

`.` matches **any** letter. When we see `.`, try **every child** (DFS). Otherwise follow the single matching child.

In [1]:
class Node:
    def __init__(self):
        self.children = {}
        self.is_end = False

class WordDictionary:
    def __init__(self):
        self.root = Node()

    def add_word(self, word):
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, Node())
        node.is_end = True

    def search(self, word):
        def dfs(i, node):
            if i == len(word):
                return node.is_end
            ch = word[i]
            if ch == ".":
                return any(dfs(i + 1, child) for child in node.children.values())
            return ch in node.children and dfs(i + 1, node.children[ch])
        return dfs(0, self.root)

wd = WordDictionary()
for w in ["bad", "dad", "mad"]:
    wd.add_word(w)
print(wd.search("pad"), wd.search("bad"), wd.search(".ad"), wd.search("b.."))   # False True True True

False True True True


## 2. Replace Words

Replace each word in a sentence by its **shortest root** from a dictionary. Insert roots into a trie. For each word walk down and stop at the **first** `is_end`.

In [2]:
def replace_words(dictionary, sentence):
    root = Node()
    for w in dictionary:
        node = root
        for ch in w:
            node = node.children.setdefault(ch, Node())
        node.is_end = True

    def shortest_root(word):
        node = root
        for i, ch in enumerate(word):
            if ch not in node.children:
                return word
            node = node.children[ch]
            if node.is_end:
                return word[:i + 1]
        return word

    return " ".join(shortest_root(w) for w in sentence.split())

print(replace_words(["cat", "bat", "rat"], "the cattle was rattled by the battery"))   # the cat was rat by the bat

the cat was rat by the bat


## 3. Longest Word in Dictionary

The longest word that can be built **one character at a time**, where every prefix is also a word. In the trie, only walk through nodes where `is_end` is `True`. Go through children in alphabetical order and keep the first longest word.

In [3]:
def longest_word(words):
    root = Node()
    for w in words:
        node = root
        for ch in w:
            node = node.children.setdefault(ch, Node())
        node.is_end = True
    best = ""
    def dfs(node, path):
        nonlocal best
        if len(path) > len(best):
            best = "".join(path)
        for ch in sorted(node.children):
            child = node.children[ch]
            if child.is_end:                         # only extend through real words
                path.append(ch)
                dfs(child, path)
                path.pop()
    dfs(root, [])
    return best

print(longest_word(["w", "wo", "wor", "worl", "world"]), longest_word(["a", "banana", "app", "appl", "ap", "apply", "apple"]))   # world apple

world apple


## 4. Longest Common Prefix

Insert all words, then walk from the root **while the node has exactly one child and no word ends there**. (A simpler way is comparing first and last after sorting, but the trie extends to many other questions.)

In [4]:
def longest_common_prefix(words):
    root = Node()
    for w in words:
        node = root
        for ch in w:
            node = node.children.setdefault(ch, Node())
        node.is_end = True
    prefix, node = [], root
    while len(node.children) == 1 and not node.is_end:
        ch, node = next(iter(node.children.items()))
        prefix.append(ch)
    return "".join(prefix)

print(longest_common_prefix(["flower", "flow", "flight"]), repr(longest_common_prefix(["dog", "racecar", "car"])))   # fl ''

fl ''


## 5. Map Sum Pairs (design)

`insert(key, val)` (overwrites) and `sum(prefix)` = sum of values of all keys with that prefix. Store a `total` in every node = sum of values of words passing through. On insert, apply the **difference** from the old value to every node on the path.

In [5]:
class MapSum:
    def __init__(self):
        self.root = {"total": 0, "kids": {}}
        self.values = {}

    def insert(self, key, val):
        delta = val - self.values.get(key, 0)
        self.values[key] = val
        node = self.root
        for ch in key:
            node = node["kids"].setdefault(ch, {"total": 0, "kids": {}})
            node["total"] += delta

    def sum(self, prefix):
        node = self.root
        for ch in prefix:
            if ch not in node["kids"]:
                return 0
            node = node["kids"][ch]
        return node["total"]

ms = MapSum()
ms.insert("apple", 3)
print(ms.sum("ap"))   # 3
ms.insert("app", 2)
print(ms.sum("ap"))   # 5
ms.insert("apple", 2)
print(ms.sum("ap"))   # 4

3
5
4


## 6. Interview thinking

**Signals:** "prefix", "words starting with", "wildcard", "dictionary of words", "shortest/longest word with a property".

**Say:** "A trie makes each query O(L). For the wildcard I branch only at `.` so the worst case is exponential in the number of dots but small in practice."

**Common mistakes:** forgetting `is_end` in Replace Words (stop at the first word end), applying the full value instead of the delta in Map Sum, not handling the empty prefix.

## Quick Revision

- Wildcard `.`: DFS over all children.
- Replace words: stop at the first `is_end`.
- Longest buildable word: only walk `is_end` nodes.
- LCP: walk while one child and no word ends.
- Map sum: node totals updated by the delta.